In [1]:
import tensorflow as tf 
from tensorflow.keras.applications import ResNet50 
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
# Memuat model dasar ResNet50 (Transfer Learning)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Membekukan semua lapisan dasar agar tidak berubah
for layer in base_model.layers:
    layer.trainable = False


In [3]:
# Menambahkan 'Kepala' Baru untuk 2 kelas (Kucing dan Anjing)
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dense(2, activation='softmax')(x)  # 2 Kelas (Kucing, Anjing)

# Menyatukan model
model = Model(inputs=base_model.input, outputs=x)

# Compile model awal
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# model.summary() # Hilangkan komentar jika ingin melihat arsitektur


In [4]:
# --- PERSIAPAN DATA ---
# Pastikan Anda punya folder gambar! 
# Ganti tulisan 'NAMA_FOLDER_DATASET_ANDA' dengan lokasi folder asli di komputer Anda.
dataset_path = r'd:\AI_Engeener\DeepLearn\dataset'

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    dataset_path, 
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path, 
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


Found 16 images belonging to 2 classes.
Found 4 images belonging to 2 classes.


In [5]:
# Membuka 10 lapisan terakhir untuk Fine-Tuning
for layer in base_model.layers[-10:]:
    layer.trainable = True

# Compile ulang dengan learning rate sangat kecil
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Mulai Melatih AI (Hilangkan tanda komentar # di bawah ini jika dataset_path sudah benar!)
history = model.fit(train_data, epochs=5, validation_data=val_data)


Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - accuracy: 0.5000 - loss: 0.6997 - val_accuracy: 0.5000 - val_loss: 0.7016
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 986ms/step - accuracy: 0.5000 - loss: 0.8665 - val_accuracy: 0.5000 - val_loss: 0.7174
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 944ms/step - accuracy: 0.5000 - loss: 0.6878 - val_accuracy: 0.5000 - val_loss: 0.7392
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 987ms/step - accuracy: 0.5000 - loss: 0.7277 - val_accuracy: 0.5000 - val_loss: 0.7469
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 948ms/step - accuracy: 0.5000 - loss: 0.7213 - val_accuracy: 0.5000 - val_loss: 0.7431


In [ ]:
import matplotlib.pyplot as plt

if 'history' in locals():
    plt.figure(figsize=(12, 4))
    
    # Plot Akurasi
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Grafik Akurasi')
    plt.xlabel('Epoch')
    plt.ylabel('Akurasi')
    plt.legend()

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Grafik Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("Jalankan model.fit terlebih dahulu untuk menghasilkan history.")
